# Exploratory Data Analysis (EDA)

En esta fase se cargarán los datos originales procedentes de Hugging Face u otras fuentes. Nuestro objetivo es **comprender mejor nuestro dataset**: visualizaremos su estructura, analizaremos la distribución de las etiquetas (`passed`/`failed`) y revisaremos ejemplos concretos para entender el contexto de las conversaciones de seguridad (como Jailbreaks y Prompt Injection).

Con este conocimiento estaremos listos para preparar los datos adecuadamente y llevar a cabo el *fine-tuning* de nuestro modelo juez (Prometheus).

In [1]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from dotenv import load_dotenv

load_dotenv()

# Añadir src al path
sys.path.append(os.path.abspath(os.path.join('../src')))

from data_utils import *

In [ ]:
# Usamos el dataset completo de entrenamiento para el análisis
DATA_INPUT_FILENAME = os.getenv("DATA_INPUT_FILENAME", "../data/dataset_train_ds.json")
print(f"Analizando: {DATA_INPUT_FILENAME}")

In [ ]:
df_raw = load_data(DATA_INPUT_FILENAME)
print(f"Total registros: {len(df_raw)}")
df_raw.head()

In [ ]:
df_raw.iloc[0]

In [ ]:
print(f"Total registros: {len(df_raw)}")
print(df_raw.info())

In [ ]:
df_raw.groupby('user_id').count()

## Distribución de Veredictos

A continuación, visualizamos cuántos ejemplos tenemos de tipo `passed` (el modelo ha rehusado la solicitud maliciosa con éxito) y `failed` (el modelo ha caído en la trampa adversaria).
Esto es clave para entender y balancear nuestro dataset antes del fine-tuning.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(y='verdict', data=df_raw)
plt.title('Distribución de Verdicts')
plt.show()

In [ ]:
# Ver ejemplos aleatorios
sample = df_raw.iloc[3]
print("Question:", sample['raw']['messages'])
print("\nReference Answer:", sample['raw']['messages'][1]['content'])
print("\nProposal Answer:", sample.get('proposed_answer'))
print("\nVerdict:", sample['verdict'])

In [ ]:
import os
os.makedirs("../output", exist_ok=True)

from data_utils import prepare_dataset, filter_high_quality
df = prepare_dataset(df_raw)
df_clean = filter_high_quality(df)
print(f"df: {len(df)} muestras | df_clean: {len(df_clean)} muestras de calidad")

In [ ]:
# ── Análisis de longitud de conversaciones ─────────────────────────────────────
df_clean["n_turns"] = df_clean["history"].apply(
    lambda h: len(h) // 2 + 1 if isinstance(h, list) else 1
)
df_clean["answer_len"] = df_clean["answer"].str.len()
df_clean["question_len"] = df_clean["question"].str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Turnos de conversación
axes[0].hist(df_clean["n_turns"], bins=range(1, df_clean["n_turns"].max() + 2), 
             color="#8b5cf6", edgecolor="white")
axes[0].set_title("Distribución de turnos de conversación")
axes[0].set_xlabel("Nº de turnos"); axes[0].set_ylabel("Frecuencia")

# Longitud de la respuesta del modelo
for verdict, color, label in [("1", "#22c55e", "Passed"), ("0", "#ef4444", "Failed")]:
    subset = df_clean[df_clean["verdict"] == verdict]["answer_len"]
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=label)
axes[1].set_title("Longitud de la respuesta del modelo")
axes[1].set_xlabel("Nº caracteres"); axes[1].legend()

# Longitud por veredicto (boxplot)
df_clean.boxplot(column="answer_len", by="verdict", ax=axes[2], 
                 patch_artist=True)
axes[2].set_title("Longitud respuesta por veredicto")
axes[2].set_xlabel("Verdict (0=failed, 1=passed)")
axes[2].set_ylabel("Nº caracteres")
plt.suptitle("")

plt.tight_layout()
plt.savefig("../output/conversation_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nEstadísticas de longitud de respuesta:")
print(df_clean.groupby("verdict")["answer_len"].describe().round(0))
print(f"\nMulti-turno (>1 turno): {(df_clean['n_turns'] > 1).sum()} muestras ({(df_clean['n_turns'] > 1).mean():.1%})")

In [ ]:
# ── Análisis por categoría: distribución de veredictos ────────────────────────
df_clean = filter_high_quality(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de categorías
cat_counts = df_clean["category_name"].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color="#3b82f6")
axes[0].set_title("Distribución de categorías (muestras de calidad)")
axes[0].set_xlabel("Nº muestras")
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 0.1, i, str(v), va='center', fontweight='bold')

# Veredicto por categoría
cat_verdict = df_clean.groupby(["category_name", "verdict"]).size().unstack(fill_value=0)
cat_verdict.plot(kind="barh", ax=axes[1], color={"0": "#ef4444", "1": "#22c55e"})
axes[1].set_title("Veredictos por categoría")
axes[1].set_xlabel("Nº muestras")
axes[1].legend(["Failed (0)", "Passed (1)"], title="Verdict")

plt.tight_layout()
plt.savefig("../output/category_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen
summary = df_clean.groupby("category_name").agg(
    total=("verdict", "count"),
    passed=("verdict", lambda x: (x == "1").sum()),
    failed=("verdict", lambda x: (x == "0").sum()),
    pass_rate=("verdict", lambda x: f"{(x == '1').mean():.1%}")
).sort_values("total", ascending=False)
print("\nResumen por categoría:")
print(summary.to_string())

In [ ]:
# ── Análisis de calidad de datos ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. val_context_bool
val_ctx = df["val_context_bool"].value_counts()
axes[0].bar(val_ctx.index.astype(str), val_ctx.values, color=["#22c55e", "#ef4444"])
axes[0].set_title("val_context_bool\n(¿Conversación relevante al reto?)")
axes[0].set_xlabel(""); axes[0].set_ylabel("Nº muestras")
for i, v in enumerate(val_ctx.values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# 2. val_stop_reason
stop = df["val_stop_reason"].fillna("None/OK").value_counts()
axes[1].bar(stop.index, stop.values, color=["#3b82f6", "#f59e0b", "#ef4444", "#6b7280"][:len(stop)])
axes[1].set_title("val_stop_reason\n(Motivo de parada en validación)")
axes[1].tick_params(axis='x', rotation=25)
for i, v in enumerate(stop.values):
    axes[1].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# 3. Veredicto por calidad de contexto
df_cross = df.groupby(["val_context_bool", "verdict"]).size().unstack(fill_value=0)
df_cross.plot(kind="bar", ax=axes[2], color=["#ef4444", "#22c55e"])
axes[2].set_title("Distribución de veredictos\npor calidad de contexto")
axes[2].set_xlabel("val_context_bool")
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title="Verdict")

plt.tight_layout()
plt.savefig("../output/data_quality_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMuestras con contexto INVÁLIDO (serán excluidas del entrenamiento): {(df['val_context_bool'] == False).sum()}")
print(f"Muestras con contexto VÁLIDO (usadas para entrenamiento): {(df['val_context_bool'] == True).sum()}")

## Análisis de Calidad del Dataset

Inspeccionamos los campos de validación para identificar muestras problemáticas que deberíamos excluir del entrenamiento.

In [ ]:
from data_utils import prepare_dataset, filter_high_quality

# Preparar el dataset con todas las columnas de calidad
df = prepare_dataset(df_raw)
print(f"Columnas disponibles: {list(df.columns)}")
print(f"\nTotal muestras: {len(df)}")
